# Hassaniya ASR — Whisper-Medium Fine-Tuning

Fine-tunes `openai/whisper-medium` on the Hassaniya dataset built by `Preprocessing.ipynb`, then scores it once on the held-out test set.

Runs on Kaggle (2× T4). Checkpoints are chosen by **validation WER**; the test set is only used at the end.

## 1. Setup
Install the two packages Kaggle doesn't ship with and check the GPUs.

In [2]:
!pip install -q -U bitsandbytes jiwer
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

index, name, memory.total [MiB]
0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB


## 2. Dataset
Finds the uploaded dataset folder (the one containing `manifest.json`) and shows which version and split sizes this run will use.

In [3]:
import glob, json, os

manifests = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert len(manifests) == 1, manifests
DATA_DIR = os.path.dirname(manifests[0])

manifest = json.load(open(manifests[0], encoding="utf-8"))
print("Data dir:", DATA_DIR)
print("Version:", manifest["dataset_version"])
for split, info in manifest["splits"].items():
    print(f"{split:<11} {info['rows']:>5} rows  {info['hours']:.2f} h  {info['rows_per_source']}")

Data dir: /kaggle/input/datasets/hamidameer/hassaniya-asr
Version: v2
train        2525 rows  2.23 h  {'datasetA': 702, 'datasetB': 1579, 'datasetC': 244}
validation    268 rows  0.22 h  {'datasetA': 92, 'datasetB': 145, 'datasetC': 31}
test          289 rows  0.23 h  {'datasetA': 102, 'datasetB': 168, 'datasetC': 19}


## 3. Training script
The pipeline lives in `train_whisper.py` (same file as in the repo). It is written to disk in pieces below: the first cell creates the file and each following cell appends to it (`%%writefile -a`), so **run all of section 3, in order**, before the next steps.

### 3.1 Imports and constants
Libraries plus the fixed settings: 16 kHz audio, Arabic, transcription task.

In [4]:
%%writefile /kaggle/working/train_whisper.py
"""Fine-tune Whisper on the Hassaniya ASR dataset produced by Preprocessing.ipynb.

Single-GPU:  python train_whisper.py --data_dir DIR
Two GPUs:    accelerate launch --num_processes 2 --mixed_precision fp16 train_whisper.py --grad_accum 4
Pre-flight:  python train_whisper.py --dry_run   (checks labels + loss on a few batches, no training)
Resume:      same command plus --resume   (continues from the newest checkpoint in --output_dir)
Test only:   python train_whisper.py --eval_only --model_id <saved model folder or hub id>
             (e.g. the zero-shot baseline: --model_id openai/whisper-medium)

Checkpoints are selected on validation WER; the test set is scored once, at the end.
"""

import argparse
import csv
import glob
import io
import json
import os
import re
import unicodedata
from dataclasses import dataclass
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import soundfile as sf
import torch
from datasets import Dataset
from jiwer import cer, wer
from transformers import (
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)
import transformers

SAMPLING_RATE = 16000
RUNAWAY_LENGTH_RATIO = 2.0  # a prediction with more than 2x the reference's words counts as a runaway
LANGUAGE = "arabic"
TASK = "transcribe"

Writing /kaggle/working/train_whisper.py


### 3.2 Command-line options
Every setting can be overridden from the command line. Generation is guarded against looping output: at most 256 tokens, and no repeated 5-token sequence.

In [5]:
%%writefile -a /kaggle/working/train_whisper.py

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--model_id", default="openai/whisper-medium")
    p.add_argument("--data_dir", default=None,
                   help="Folder with manifest.json + parquet files; auto-found under /kaggle/input if omitted")
    p.add_argument("--output_dir", default="/kaggle/working/whisper-medium-hassaniya")
    p.add_argument("--run_name", default=None)
    p.add_argument("--epochs", type=float, default=5)
    p.add_argument("--lr", type=float, default=1e-5)
    p.add_argument("--warmup_steps", type=int, default=50)
    p.add_argument("--batch_size", type=int, default=1)
    p.add_argument("--eval_batch_size", type=int, default=4)
    p.add_argument("--grad_accum", type=int, default=8)
    p.add_argument("--early_stopping_patience", type=int, default=2)
    # Longest label in dataset v2 is 223 tokens; a tighter cap limits the damage of a looping generation
    p.add_argument("--generation_max_length", type=int, default=256)
    p.add_argument("--no_repeat_ngram_size", type=int, default=5,
                   help="Block any repeated n-token sequence during generation (0 = off); stops looping output")
    p.add_argument("--num_beams", type=int, default=1)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--resume", action="store_true")
    p.add_argument("--dry_run", action="store_true")
    p.add_argument("--eval_only", action="store_true",
                   help="Score --model_id (hub id or a saved folder such as <output_dir>/best) on the test set; no training")
    return p.parse_args()

Appending to /kaggle/working/train_whisper.py


### 3.3 Text normalization
The same `normalize_arabic` as in the data-prep notebook, applied to references and predictions before scoring.

In [6]:
%%writefile -a /kaggle/working/train_whisper.py

# Identical copy of normalize_arabic in Preprocessing.ipynb - keep the two in sync.
def normalize_arabic(text):
    if text is None:
        return ""

    text = unicodedata.normalize("NFKC", str(text))

    # Remove Arabic diacritics
    text = re.sub(r"[ؐ-ًؚ-ٰٟۖ-ۭ]", "", text)

    # Normalize alef variants
    text = re.sub(r"[أإآٱ]", "ا", text)

    # Alef maqsura -> ya
    text = text.replace("ى", "ي")

    # Remove tatweel
    text = text.replace("ـ", "")

    # Remove punctuation (Arabic ، ؛ ؟ included; letters, digits and gaf stay)
    text = re.sub(r"[^\w\s]|_", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

Appending to /kaggle/working/train_whisper.py


### 3.4 Data loading
Reads the parquet splits listed in `manifest.json` and checks their row counts. `WhisperTransform` turns WAV bytes into log-mel features and text into token labels on the fly, with the `<|ar|><|transcribe|>` prefix.

In [7]:
%%writefile -a /kaggle/working/train_whisper.py

def find_data_dir(data_dir):
    if data_dir:
        return data_dir
    manifests = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
    if len(manifests) != 1:
        raise SystemExit(f"Pass --data_dir; found {len(manifests)} manifest.json files: {manifests}")
    return os.path.dirname(manifests[0])


def load_split(data_dir, manifest, split):
    path = os.path.join(data_dir, manifest["splits"][split]["file"])
    ds = Dataset.from_parquet(path)
    expected = manifest["splits"][split]["rows"]
    if len(ds) != expected:
        raise SystemExit(f"{split}: {len(ds)} rows on disk, manifest says {expected}")
    return ds


class WhisperTransform:
    """Decode WAV bytes -> log-mel features, text -> labels with the full Whisper prefix.

    A module-level class (not a closure) so dataloader workers can pickle it on Windows too.
    """

    def __init__(self, processor):
        self.processor = processor

    def __call__(self, batch):
        features, labels = [], []
        for audio, text in zip(batch["audio"], batch["text"]):
            waveform, sr = sf.read(io.BytesIO(audio["bytes"]), dtype="float32")
            if sr != SAMPLING_RATE or waveform.ndim != 1:
                raise ValueError(f"Expected 16 kHz mono, got sr={sr} shape={waveform.shape}")
            features.append(
                self.processor.feature_extractor(waveform, sampling_rate=SAMPLING_RATE).input_features[0]
            )
            labels.append(self.processor.tokenizer(text).input_ids)
        return {"input_features": features, "labels": labels}

Appending to /kaggle/working/train_whisper.py


### 3.5 Batching
Pads labels with `-100` so padding is ignored by the loss, and drops the leading `<|startoftranscript|>` because the model adds it itself.

In [8]:
%%writefile -a /kaggle/working/train_whisper.py

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: WhisperProcessor
    decoder_start_token_id: int

    def __call__(self, features):
        batch = self.processor.feature_extractor.pad(
            [{"input_features": f["input_features"]} for f in features],
            return_tensors="pt",
        )

        labels_batch = self.processor.tokenizer.pad(
            [{"input_ids": f["labels"]} for f in features],
            return_tensors="pt",
        )
        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)

        # The model prepends <|startoftranscript|> itself when it shifts labels right.
        # Compare against decoder_start_token_id (50258), not tokenizer.bos_token_id (50257).
        if (labels[:, 0] == self.decoder_start_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

Appending to /kaggle/working/train_whisper.py


### 3.6 Metrics
WER and CER on normalized text — used to pick the best checkpoint on the validation set.

In [9]:
%%writefile -a /kaggle/working/train_whisper.py

def make_compute_metrics(processor):
    tokenizer = processor.tokenizer

    def compute_metrics(pred):
        pred_ids = pred.predictions
        label_ids = np.where(pred.label_ids == -100, tokenizer.pad_token_id, pred.label_ids)
        pred_ids = np.where(pred_ids == -100, tokenizer.pad_token_id, pred_ids)

        # Labels were normalized at prep time; normalizing again is a no-op for refs
        preds = [normalize_arabic(s) for s in tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
        refs = [normalize_arabic(s) for s in tokenizer.batch_decode(label_ids, skip_special_tokens=True)]

        return {
            "wer": 100 * wer(refs, preds),
            "cer": 100 * cer(refs, preds),
        }

    return compute_metrics


def environment_info():
    gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    return {
        "gpus": gpus,
        "world_size": int(os.environ.get("WORLD_SIZE", 1)),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
    }

Appending to /kaggle/working/train_whisper.py


### 3.7 Pre-flight check
Used by `--dry_run`: checks the label prefix and the pretrained loss on a few clips before any GPU hours are spent.

In [10]:
%%writefile -a /kaggle/working/train_whisper.py

def dry_run(model, processor, collator, train_ds):
    """One-process check that labels carry the right prefix and the loss is sane before a long run."""
    sample = train_ds[0]
    print("Label prefix:", processor.tokenizer.convert_ids_to_tokens(sample["labels"][:4]))

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device).eval()
    batch = collator([train_ds[i] for i in range(4)])
    print("Collated label start:", processor.tokenizer.convert_ids_to_tokens(batch["labels"][0][:3].tolist()))

    with torch.no_grad():
        out = model(
            input_features=batch["input_features"].to(device),
            labels=batch["labels"].to(device),
        )
    print(f"Pretrained loss on 4 samples: {out.loss.item():.3f}  (expect roughly 1-6; far higher means malformed labels)")

    generated = model.generate(batch["input_features"][:2].to(device), language=LANGUAGE, task=TASK, max_new_tokens=60)
    for ids in generated:
        print("Zero-shot:", processor.tokenizer.decode(ids, skip_special_tokens=True))

Appending to /kaggle/working/train_whisper.py


### 3.8 Main — setup
Loads the processor, model and data splits. `--dry_run` and `--eval_only` finish here.

In [11]:
%%writefile -a /kaggle/working/train_whisper.py

def main():
    args = parse_args()
    run_name = args.run_name or datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    data_dir = find_data_dir(args.data_dir)

    with open(os.path.join(data_dir, "manifest.json"), encoding="utf-8") as f:
        manifest = json.load(f)

    processor = WhisperProcessor.from_pretrained(args.model_id, language=LANGUAGE, task=TASK)

    model = WhisperForConditionalGeneration.from_pretrained(args.model_id)
    model.generation_config.language = LANGUAGE
    model.generation_config.task = TASK
    model.generation_config.forced_decoder_ids = None
    model.config.forced_decoder_ids = None
    # Whisper's forward takes **kwargs, so Trainer assumes it averages loss over num_items_in_batch
    # and skips the grad-accum division; it doesn't (plain mean CE), so loss would be inflated by grad_accum x GPUs.
    model.accepts_loss_kwargs = False
    # Applies to validation generation (Trainer) and test scoring alike
    model.generation_config.no_repeat_ngram_size = args.no_repeat_ngram_size

    train_ds = load_split(data_dir, manifest, "train")
    val_ds = load_split(data_dir, manifest, "validation")
    test_ds = load_split(data_dir, manifest, "test")
    # Materialize now: datasets>=4 returns a lazy Column that would go through the transform later
    test_sources = list(test_ds["source"])
    test_texts = list(test_ds["text"])
    for ds in (train_ds, val_ds, test_ds):
        ds.set_transform(WhisperTransform(processor))

    collator = DataCollatorSpeechSeq2SeqWithPadding(
        processor=processor,
        decoder_start_token_id=model.config.decoder_start_token_id,
    )

    if args.dry_run:
        print(f"Dataset {manifest['dataset_version']} from {data_dir}: "
              f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")
        dry_run(model, processor, collator, train_ds)
        return

    if args.eval_only:
        predictions, metrics = score_test(model, processor, collator, test_ds, test_texts, test_sources, args)
        write_results(args, run_name, manifest, predictions, metrics, training_info={})
        return

Appending to /kaggle/working/train_whisper.py


### 3.9 Main — training
Continues `main()`: trainer settings (fp16, 8-bit AdamW, gradient checkpointing, best checkpoint chosen on validation WER with early stopping), the training run, then one test-set score for the best model.

In [12]:
%%writefile -a /kaggle/working/train_whisper.py

    training_args = Seq2SeqTrainingArguments(
        output_dir=args.output_dir,
        run_name=run_name,

        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.eval_batch_size,
        gradient_accumulation_steps=args.grad_accum,

        learning_rate=args.lr,
        warmup_steps=args.warmup_steps,
        num_train_epochs=args.epochs,

        fp16=torch.cuda.is_available(),
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="adamw_bnb_8bit",

        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,

        predict_with_generate=True,
        generation_max_length=args.generation_max_length,
        generation_num_beams=args.num_beams,

        logging_strategy="steps",
        logging_steps=25,

        remove_unused_columns=False,
        dataloader_num_workers=2,
        ddp_find_unused_parameters=False,
        seed=args.seed,
        report_to=[],
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        processing_class=processor,
        compute_metrics=make_compute_metrics(processor),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=args.early_stopping_patience)],
    )

    config = {
        "run_name": run_name,
        "started_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "args": vars(args),
        "data_dir": data_dir,
        "dataset_version": manifest["dataset_version"],
        "dataset_splits": {k: {"rows": v["rows"], "md5": v["md5"]} for k, v in manifest["splits"].items()},
        "effective_batch_size": args.batch_size * args.grad_accum * int(os.environ.get("WORLD_SIZE", 1)),
        "environment": environment_info(),
    }
    if trainer.is_world_process_zero():
        os.makedirs(args.output_dir, exist_ok=True)
        with open(os.path.join(args.output_dir, f"run_config_{run_name}.json"), "w", encoding="utf-8") as f:
            json.dump(config, f, ensure_ascii=False, indent=2)
        print(json.dumps(config, ensure_ascii=False, indent=2), flush=True)

    trainer.train(resume_from_checkpoint=True if args.resume else None)

    best_dir = os.path.join(args.output_dir, "best")
    trainer.save_model(best_dir)
    if not trainer.is_world_process_zero():
        return
    processor.save_pretrained(best_dir)

    # Score the test set once, with the checkpoint chosen on validation WER.
    # Done on rank 0 only, in dataset order, so every prediction lines up with its test clip
    # (outputs gathered from trainer.predict across GPUs are not guaranteed to match the test set row for row).
    predictions, metrics = score_test(
        trainer.accelerator.unwrap_model(trainer.model), processor, collator, test_ds, test_texts, test_sources, args
    )

    state = trainer.state
    best_val = next(
        (log for log in state.log_history if log.get("eval_wer") == state.best_metric), {}
    )
    training_info = {
        "best_checkpoint": state.best_model_checkpoint,
        "epochs_trained": round(state.epoch, 2),
        "effective_batch_size": config["effective_batch_size"],
        "val_wer": best_val.get("eval_wer"),
        "val_cer": best_val.get("eval_cer"),
    }
    write_results(args, run_name, manifest, predictions, metrics, training_info)

Appending to /kaggle/working/train_whisper.py


### 3.10 Test scoring
Generates test transcripts on one GPU, in dataset order, and computes overall and per-source WER/CER. It also counts *runaway* clips (output more than twice the reference length) and the WER without them.

In [13]:
%%writefile -a /kaggle/working/train_whisper.py

def score_test(model, processor, collator, test_ds, test_texts, test_sources, args):
    """Generate test transcripts on a single device, in dataset order, and score them."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device).eval()

    preds = []
    for start in range(0, len(test_ds), args.eval_batch_size):
        batch = collator([test_ds[i] for i in range(start, min(start + args.eval_batch_size, len(test_ds)))])
        with torch.no_grad(), torch.autocast(device_type=device, dtype=torch.float16, enabled=device == "cuda"):
            ids = model.generate(
                batch["input_features"].to(device),
                language=LANGUAGE,
                task=TASK,
                max_length=args.generation_max_length,
                num_beams=args.num_beams,
            )
        preds.extend(normalize_arabic(s) for s in processor.tokenizer.batch_decode(ids, skip_special_tokens=True))
        print(f"test {min(start + args.eval_batch_size, len(test_ds))}/{len(test_ds)}", flush=True)

    refs = [normalize_arabic(t) for t in test_texts]
    predictions = pd.DataFrame({"source": test_sources, "reference": refs, "prediction": preds})

    # Runaway generations (looping output) inflate WER on a few clips; count them so runs can be compared
    length_ratio = predictions["prediction"].str.split().str.len() / predictions["reference"].str.split().str.len()
    runaway = predictions[length_ratio > RUNAWAY_LENGTH_RATIO]
    kept = predictions[length_ratio <= RUNAWAY_LENGTH_RATIO]

    metrics = {
        "test_wer": 100 * wer(refs, preds),
        "test_cer": 100 * cer(refs, preds),
        "test_per_source": {
            source: {"rows": len(g), "wer": 100 * wer(list(g["reference"]), list(g["prediction"])),
                     "cer": 100 * cer(list(g["reference"]), list(g["prediction"]))}
            for source, g in predictions.groupby("source")
        },
        "test_runaway_clips": len(runaway),
        "test_runaway_per_source": runaway["source"].value_counts().to_dict(),
        "test_wer_without_runaway": 100 * wer(list(kept["reference"]), list(kept["prediction"])),
    }
    return predictions, metrics

Appending to /kaggle/working/train_whisper.py


### 3.11 Saving results
Writes the predictions CSV and results JSON, and appends one row per run to `experiments.csv`.

In [14]:
%%writefile -a /kaggle/working/train_whisper.py

def write_results(args, run_name, manifest, predictions, metrics, training_info):
    os.makedirs(args.output_dir, exist_ok=True)
    predictions.to_csv(os.path.join(args.output_dir, f"test_predictions_{run_name}.csv"), index=False, encoding="utf-8")

    results = {
        "run_name": run_name,
        "finished_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "model_id": args.model_id,
        "eval_only": args.eval_only,
        **training_info,
        **metrics,
    }
    with open(os.path.join(args.output_dir, f"results_{run_name}.json"), "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(json.dumps(results, ensure_ascii=False, indent=2))

    log_path = os.path.join(args.output_dir, "experiments.csv")
    row = {
        "run_name": run_name,
        "model_id": args.model_id,
        "dataset_version": manifest["dataset_version"],
        "gpus": "+".join(environment_info()["gpus"]),
        "effective_batch_size": training_info.get("effective_batch_size"),
        "lr": None if args.eval_only else args.lr,
        "epochs_trained": training_info.get("epochs_trained"),
        "best_checkpoint": training_info.get("best_checkpoint"),
        "val_wer": training_info.get("val_wer"),
        "test_wer": metrics["test_wer"],
        "test_cer": metrics["test_cer"],
        "test_runaway_clips": metrics["test_runaway_clips"],
        "generation_max_length": args.generation_max_length,
        "no_repeat_ngram_size": args.no_repeat_ngram_size,
    }
    new_file = not os.path.exists(log_path)
    with open(log_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row))
        if new_file:
            writer.writeheader()
        writer.writerow(row)


if __name__ == "__main__":
    main()

Appending to /kaggle/working/train_whisper.py


## 4. Pre-flight check
Loads the pretrained model and checks one small batch before committing GPU hours:
- labels start with `<|startoftranscript|><|ar|><|transcribe|><|notimestamps|>`
- the loss is in a normal range (roughly 1–6)

In [15]:
!python /kaggle/working/train_whisper.py --data_dir "$DATA_DIR" --dry_run

preprocessor_config.json: 185kB [00:00, 211MB/s]
config.json: 1.99kB [00:00, 8.83MB/s]
tokenizer_config.json: 283kB [00:00, 295MB/s]
vocab.json: 836kB [00:00, 22.5MB/s]
tokenizer.json: 2.48MB [00:00, 145MB/s]
merges.txt: 494kB [00:00, 98.9MB/s]
normalizer.json: 52.7kB [00:00, 85.0MB/s]
added_tokens.json: 34.6kB [00:00, 70.4MB/s]
special_tokens_map.json: 2.19kB [00:00, 8.02MB/s]
model.safetensors: 100%|████████████████████| 3.06G/3.06G [00:06<00:00, 491MB/s]
Loading weights: 100%|█| 947/947 [00:00<00:00, 1919.11it/s, Materializing param=
generation_config.json: 3.75kB [00:00, 10.7MB/s]
Generating train split: 2525 examples [00:09, 252.67 examples/s]
Generating train split: 268 examples [00:00, 556.44 examples/s]
Generating train split: 289 examples [00:00, 580.61 examples/s]
Dataset v2 from /kaggle/input/datasets/hamidameer/hassaniya-asr: train=2525 val=268 test=289
Label prefix: ['<|startoftranscript|>', '<|ar|>', '<|transcribe|>', '<|notimestamps|>']
Collated label start: ['<|ar|>', '

## 5. Train
Uses both T4s. Effective batch size = 1 per GPU × 2 GPUs × 4 accumulation steps = 8.

Run the notebook with **Save Version → Save & Run All** so the output (models, predictions, logs) is kept after the session ends. Interactive sessions delete `/kaggle/working` when they stop.

If a run is cut short, re-run this cell with `--resume` added to continue from the latest checkpoint (same session only).

In [16]:
!accelerate launch --num_processes 2 --mixed_precision fp16 \
    /kaggle/working/train_whisper.py --data_dir "$DATA_DIR" --grad_accum 4

The following values were not passed to `accelerate launch` and had defaults used instead:
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Loading weights:  70%|▋| 660/947 [00:00<00:00, 1362.07it/s, Materializing param=
Loading weights: 100%|█| 947/947 [00:00<00:00, 1599.22it/s, Materializing param=
{
  "run_name": "20261005-143735",
  "started_utc": "2026-10-05T14:37:43+00:00",
  "args": {
    "model_id": "openai/whisper-medium",
    "data_dir": "/kaggle/input/datasets/hamidameer/hassaniya-asr",
    "output_dir": "/kaggle/working/whisper-medium-hassaniya",
    "run_name": null,
    "epochs": 5,
    "lr": 1e-05,
    "warmup_steps": 50,
    "batch_size": 1,
    "eval_batch_size": 4,
    "grad_accum": 4,
    "early_stoppin

## 5b. Score a saved model (no training)
Scores a model on the test set only — about 5 minutes on one GPU. Run these cells on their own (not with *Run All*, which would retrain).

**Fine-tuned model from an earlier run:** add that run's notebook output as an input (*Add Input → Your Work → Notebooks*); the cell below finds its `best/` folder.

In [17]:
best_dirs = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/best/model.safetensors", recursive=True)]

if best_dirs:
    BEST_DIR = best_dirs[0]
    print("Scoring:", BEST_DIR)
    !python /kaggle/working/train_whisper.py --data_dir "$DATA_DIR" --eval_only --model_id "$BEST_DIR" --run_name finetuned-rescore
else:
    print("No earlier run attached as input - skipping (the training run above already scored its best model).")

No earlier run attached as input - skipping (the training run above already scored its best model).


**Zero-shot baseline:** the pretrained model with no fine-tuning, on the same test set — the reference point for how much fine-tuning helped.

In [18]:
!python /kaggle/working/train_whisper.py --data_dir "$DATA_DIR" --eval_only --model_id openai/whisper-medium --run_name zero-shot

Loading weights: 100%|█| 947/947 [00:00<00:00, 2004.27it/s, Materializing param=
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameteri

## 6. Results
The experiment log (one row per run) and the test scores of every run in this session, overall and per source dataset.

In [19]:
import pandas as pd

OUTPUT_DIR = "/kaggle/working/whisper-medium-hassaniya"

# One row per run: training runs, re-scores and the zero-shot baseline
display(pd.read_csv(f"{OUTPUT_DIR}/experiments.csv"))

for path in sorted(glob.glob(f"{OUTPUT_DIR}/results_*.json"), key=os.path.getmtime):
    res = json.load(open(path, encoding="utf-8"))
    print(f"\n=== {res['run_name']}  ({res['model_id']}) ===")
    print(f"test WER {res['test_wer']:.2f}%  |  test CER {res['test_cer']:.2f}%"
          f"  |  runaway clips {res.get('test_runaway_clips', 'n/a')}")
    for source, m in res["test_per_source"].items():
        print(f"  {source:<9} {m['rows']:>4} clips  WER {m['wer']:.2f}%  CER {m['cer']:.2f}%")

# Most recent fine-tuned model (training run or re-score, not the zero-shot baseline), used for the samples below
def is_zero_shot(path):
    res = json.load(open(path, encoding="utf-8"))
    return res.get("eval_only") and res["model_id"] == "openai/whisper-medium"

finetuned = [p for p in glob.glob(f"{OUTPUT_DIR}/results_*.json") if not is_zero_shot(p)]
latest = max(finetuned, key=os.path.getmtime)

,run_name,model_id,dataset_version,gpus,effective_batch_size,lr,epochs_trained,best_checkpoint,val_wer,test_wer,test_cer,test_runaway_clips,generation_max_length,no_repeat_ngram_size
0,20261005-143735,openai/whisper-medium,v2,Tesla T4+Tesla T4,8.0,0.00001,5.0,/kaggle/working/whisper-medium-hassaniya/check...,60.620915,58.835182,22.494172,0,256,5
1,zero-shot,openai/whisper-medium,v2,Tesla T4+Tesla T4,NaN,NaN,NaN,NaN,NaN,85.538827,46.703297,0,256,5



=== 20261005-143735  (openai/whisper-medium) ===
test WER 58.84%  |  test CER 22.49%  |  runaway clips 0
  datasetA   102 clips  WER 53.51%  CER 16.16%
  datasetB   168 clips  WER 60.97%  CER 24.96%
  datasetC    19 clips  WER 49.49%  CER 17.25%

=== zero-shot  (openai/whisper-medium) ===
test WER 85.54%  |  test CER 46.70%  |  runaway clips 0
  datasetA   102 clips  WER 80.18%  CER 37.23%
  datasetB   168 clips  WER 87.87%  CER 50.99%
  datasetC    19 clips  WER 72.73%  CER 28.82%


## 7. Sample predictions
A handful of test-set transcripts from the fine-tuned model next to the reference (both normalized).

In [20]:
predictions = pd.read_csv(latest.replace("results_", "test_predictions_").replace(".json", ".csv"))

for _, row in predictions.sample(20, random_state=0).iterrows():
    print(f"[{row['source']}]")
    print("REF :", row["reference"])
    print("PRED:", row["prediction"])
    print()

[datasetB]
REF : انت كاع ما تنشافي
PRED: انت كاع ماتشافي

[datasetC]
REF : ايكد ايساعد الناس
PRED: ايكد ايساعد الناس

[datasetB]
REF : والله ياختي ذي المواصفات الي قلنالك فيها الل حقيقية ولا امكذبنا شي نعرفو عن خالك بعد مدة كاع تكذب في ذا المجال بيهللي المجالات كاملة فيه الناس تكذب وفيها الناس اللي ذا حق عندها اسمعتي يغير هاذي الوتة اللي ذا هي عندنا قايلك ان عنها وتة وافي عنها الكلام
PRED: والله يختيري المواصل و لا تكدن لك فين اللا حقيقية و لا مكذبن شي و نعرفو باط عن خالكه بعد مدة كاع تكذب ذا المجال و فيه اللي المجالات كاملة فين الناس تكذبو فين النا السلي ذا حق عنده اسمعتي يافعل معدن هذي ظرك اللي هي ظرك عنرا قالكن عنها و توف عنها الكلام

[datasetA]
REF : انعدل اسبوعين
PRED: انعدل اسبوعين

[datasetA]
REF : يتمتع بقواه العقلية مايقبل ان قاع لخلاق يكلم عنه فشي يسو شنه
PRED: يتمتع بخواه العقلية مايقبل انقال اخلاق وكلل معانها في شي يسو شنهو

[datasetA]
REF : وهو العاطيه البطاقة
PRED: انو هو العاطيه البطاقة

[datasetB]
REF : و لا جبرت طريقة نحصلو بيه الفظه
PRED: و لا يجبر الطريقة حصر لو بيه 